# Sign Language Recognition: Transformer Training

Run the cells from top to bottom. This workflow reuses the SX MediaPipe landmark extractor and transformer, caches landmarks for each image split, trains on the train split, selects the best epoch using validation accuracy, saves a checkpoint, and evaluates that checkpoint on the test split.

> **Data limitation:** the current dataset contains still images, not video. Each image supplies one landmark pose; the training dataset synthesizes a short window with small landmark jitter and scale drift. This is useful for experimenting with the transformer input pipeline, but it does not teach real signing motion.

## Step 1: Environment and settings

The expected data layout is `SuXin_TEMP/data/{train,valid,test}/<class>/*image*`. Set `USE_BODY_POSE` to match the feature layout you want; the default matches the earlier notebook's two-hand plus body-pose setup (225 values per image).

In [36]:
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / 'models' / 'SX' / 'train.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not find models/SX/train.py from the notebook working directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.SX.train import (
    SignLanguageWindowDataset,
    cache_landmarks,
    load_image_samples,
)
from models.SX.transformer import build_gesture_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = PROJECT_ROOT / 'SuXin_TEMP' / 'data'
CACHE_ROOT = PROJECT_ROOT / 'SuXin_TEMP' / 'landmark_cache_transformer'
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'sx_transformer_TwoHand_best.pt'

TWO_HANDS = True
USE_BODY_POSE = False
WINDOW_SIZE = 16
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 1e-3
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 2
FEEDFORWARD_DIM = 256
DROPOUT = 0.2
POOLING = 'cls'

if WINDOW_SIZE < 2:
    raise ValueError('The transformer needs WINDOW_SIZE >= 2.')

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Data root: {DATA_ROOT}')
print(f'Feature layout: two_hands={TWO_HANDS}, use_body_pose={USE_BODY_POSE}')

Project root: c:\Sean Su\NUS_Project\VisionSpeak
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti
Data root: c:\Sean Su\NUS_Project\VisionSpeak\SuXin_TEMP\data
Feature layout: two_hands=True, use_body_pose=False


## Step 2: Load splits and cache landmarks

MediaPipe runs once per image and writes one `features` array per image. This separate cache directory avoids confusing the transformer's cache with the older LSTM notebook cache.

In [37]:
SPLITS = ('train', 'valid', 'test')
samples_by_split = {}
for split in SPLITS:
    split_dir = DATA_ROOT / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f'Missing dataset split: {split_dir}')
    samples_by_split[split] = load_image_samples(split_dir)
    if not samples_by_split[split]:
        raise ValueError(f'No supported image files found under {split_dir}')

class_names = sorted(
    {label for split in ('train', 'valid') for _, label in samples_by_split[split]}
)
class_to_idx = {name: index for index, name in enumerate(class_names)}
unknown_test = sorted({label for _, label in samples_by_split['test']} - set(class_names))
if unknown_test:
    raise ValueError(f'test contains classes absent from train/valid: {unknown_test}')

print(f'Classes ({len(class_names)}): {class_names}')
for split in SPLITS:
    print(f'{split}: {len(samples_by_split[split])} images')

feature_size = (126 if TWO_HANDS else 63) + (99 if USE_BODY_POSE else 0)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cache_report = {}
for split in SPLITS:
    detected, missing = cache_landmarks(
        samples_by_split[split],
        CACHE_ROOT / split,
        two_hands=TWO_HANDS,
        sequence_length=1,
        use_body_pose=USE_BODY_POSE,
    )
    cache_report[split] = {'detected': detected, 'missing': missing}
    print(f'{split}: detected={detected}, no landmarks={missing}')

print(f'Landmark vector size: {feature_size}')
print(f'Cache directory: {CACHE_ROOT}')

Classes (18): ['apple', 'can', 'get', 'good', 'have', 'help', 'how', 'i', 'like', 'love', 'my', 'no', 'sorry', 'ty', 'want', 'yes', 'you', 'your']
train: 459 images
valid: 45 images
test: 23 images
train: detected=318, no landmarks=141
valid: detected=31, no landmarks=14
test: detected=21, no landmarks=2
Landmark vector size: 126
Cache directory: c:\Sean Su\NUS_Project\VisionSpeak\SuXin_TEMP\landmark_cache_transformer


## Step 3: Build transformer datasets and model

Training windows use landmark jitter and dropout plus a small smooth scale drift. Validation and test windows are not augmented. The model sees tensors shaped `(batch, window, features)`.

In [38]:
train_dataset = SignLanguageWindowDataset(
    samples_by_split['train'],
    class_to_idx,
    two_hands=TWO_HANDS,
    window_size=WINDOW_SIZE,
    cache_dir=CACHE_ROOT / 'train',
    use_body_pose=USE_BODY_POSE,
    augment=True,
)
valid_dataset = SignLanguageWindowDataset(
    samples_by_split['valid'],
    class_to_idx,
    two_hands=TWO_HANDS,
    window_size=WINDOW_SIZE,
    cache_dir=CACHE_ROOT / 'valid',
    use_body_pose=USE_BODY_POSE,
    augment=False,
)
test_dataset = SignLanguageWindowDataset(
    samples_by_split['test'],
    class_to_idx,
    two_hands=TWO_HANDS,
    window_size=WINDOW_SIZE,
    cache_dir=CACHE_ROOT / 'test',
    use_body_pose=USE_BODY_POSE,
    augment=False,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

input_size = feature_size
model_settings = {
    'architecture': 'transformer',
    'input_size': input_size,
    'num_classes': len(class_names),
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'window_size': WINDOW_SIZE,
    'sequence_length': WINDOW_SIZE,
    'two_hands': TWO_HANDS,
    'use_body_pose': USE_BODY_POSE,
    'd_model': D_MODEL,
    'num_heads': NUM_HEADS,
    'feedforward_dim': FEEDFORWARD_DIM,
    'pooling': POOLING,
    'max_tokens': WINDOW_SIZE + 1,
}
model = build_gesture_model(model_settings, num_classes=len(class_names)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

print(f'Train/valid/test: {len(train_dataset)}/{len(valid_dataset)}/{len(test_dataset)}')
print(f'Input tensor per batch: ({BATCH_SIZE}, {WINDOW_SIZE}, {input_size})')
print(f'Model parameters: {sum(parameter.numel() for parameter in model.parameters()):,}')

Train/valid/test: 459/45/23
Input tensor per batch: (16, 16, 126)
Model parameters: 286,098


## Step 4: Train for all epochs and save the best checkpoint

Training runs for all `EPOCHS`. The notebook saves a checkpoint only when validation accuracy improves, so the file retains the best validation result seen during the full run.

In [39]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.set_grad_enabled(training):
        for sequences, labels in loader:
            sequences = sequences.to(DEVICE)
            labels = labels.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(sequences)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

    denominator = max(total_samples, 1)
    return total_loss / denominator, total_correct / denominator


CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
best_val_acc = -1.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    valid_loss, valid_acc = run_epoch(valid_loader, training=False)
    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss {train_loss:.4f}, acc {train_acc:.4f} | '
        f'valid loss {valid_loss:.4f}, acc {valid_acc:.4f}'
    )

    if valid_acc > best_val_acc:
        best_val_acc = valid_acc
        best_epoch = epoch
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'class_to_idx': class_to_idx,
            'idx_to_class': {index: name for name, index in class_to_idx.items()},
            'best_val_acc': best_val_acc,
            'epoch': epoch,
            **model_settings,
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f'Saved best checkpoint: {CHECKPOINT_PATH}')

print(f'Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}')
print(f'Checkpoint: {CHECKPOINT_PATH}')

Epoch 01/100 | train loss 3.4222, acc 0.0784 | valid loss 3.2682, acc 0.1111
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_transformer_TwoHand_best.pt
Epoch 02/100 | train loss 3.0707, acc 0.0871 | valid loss 3.1326, acc 0.1111
Epoch 03/100 | train loss 2.9107, acc 0.1111 | valid loss 3.0664, acc 0.1111
Epoch 04/100 | train loss 2.8724, acc 0.1176 | valid loss 2.9803, acc 0.0889
Epoch 05/100 | train loss 2.8326, acc 0.1416 | valid loss 2.9521, acc 0.1333
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_transformer_TwoHand_best.pt
Epoch 06/100 | train loss 2.7196, acc 0.1503 | valid loss 2.8972, acc 0.0889
Epoch 07/100 | train loss 2.5534, acc 0.1874 | valid loss 3.1008, acc 0.0667
Epoch 08/100 | train loss 2.5833, acc 0.2092 | valid loss 2.8832, acc 0.2000
Saved best checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_transformer_TwoHand_best.pt
Epoch 09/100 | train loss 2.4533, acc 0.2135 | valid loss 2.6348, acc 0.2222
Save

## Step 5: Reload checkpoint and evaluate the test split

Reloading through the SX inference loader verifies that the saved architecture and landmark metadata reconstruct the model correctly.

In [40]:
from models.SX.inference import load_sx_checkpoint

best_model, checkpoint_classes, checkpoint_config = load_sx_checkpoint(CHECKPOINT_PATH, DEVICE)
if checkpoint_classes != class_to_idx:
    raise ValueError('Loaded checkpoint class mapping does not match this notebook run.')

test_loss_total = 0.0
test_correct = 0
test_total = 0
with torch.inference_mode():
    for sequences, labels in test_loader:
        sequences = sequences.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = best_model(sequences)
        loss = criterion(logits, labels)
        test_loss_total += loss.item() * labels.size(0)
        test_correct += (logits.argmax(dim=1) == labels).sum().item()
        test_total += labels.size(0)

test_loss = test_loss_total / max(test_total, 1)
test_accuracy = test_correct / max(test_total, 1)
print(f'Loaded architecture: {checkpoint_config.architecture}')
print(f'Loaded window size: {checkpoint_config.window_size}')
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f} ({test_correct}/{test_total})')
print(f'Saved checkpoint: {CHECKPOINT_PATH}')

Loaded architecture: transformer
Loaded window size: 16
Test loss: 2.2133
Test accuracy: 0.6087 (14/23)
Saved checkpoint: c:\Sean Su\NUS_Project\VisionSpeak\checkpoints\sx_transformer_TwoHand_best.pt


## Interpreting this experiment

The validation and test scores measure recognition of still-image poses under the synthetic-window setup, not recognition of movement. For dynamic word signs, collect labeled video clips with per-frame landmark arrays shaped `(frames, 126)` or `(frames, 225)`, and enumerate training samples by clip.

## Step 6: Live webcam prediction (Mac / Windows)

Choose a checkpoint in the next cell. The checkpoint loader detects whether it is a transformer or LSTM and reads the required hand/body feature layout and window size. Copy the chosen checkpoint onto the Mac and update its path if this notebook is running from a different project copy. On macOS, allow camera access for VS Code or the app running Jupyter. Press `q` or `Esc` to stop.

In [ ]:
from collections import deque
import platform

import cv2
import numpy as np
import torch

from models.SX.features import extract_rgb
from models.SX.inference import choose_device, load_sx_checkpoint
from processing.mediapipe_compat import mp

# Select one checkpoint. The default uses the checkpoint configured in Step 1.
# To switch models, assign CHECKPOINT_PATH to another compatible SX checkpoint.
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'sx_transformer_TwoHand_best.pt'
# CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'sx_transformer_TwoHandAndBody_best.pt'

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT_PATH}. Copy it to this Mac or set CHECKPOINT_PATH above.'
    )

device = choose_device('auto')
webcam_model, class_to_idx_webcam, webcam_config = load_sx_checkpoint(
    CHECKPOINT_PATH,
    device,
)
idx_to_class_webcam = {index: name for name, index in class_to_idx_webcam.items()}
print(f'Checkpoint: {CHECKPOINT_PATH.name}')
print(f'Architecture: {webcam_config.architecture}')
print(f'Feature size: {webcam_config.input_size}')
print(f'Window size: {webcam_config.window_size}')
print(f'Device: {device}')

camera_index = 0
if platform.system() == 'Darwin':
    camera = cv2.VideoCapture(camera_index, cv2.CAP_AVFOUNDATION)
else:
    camera = cv2.VideoCapture(camera_index)

if not camera.isOpened():
    camera.release()
    raise RuntimeError(
        'Could not open the webcam. On macOS, grant camera access to VS Code or the app running Jupyter.'
    )

window_name = 'VisionSpeak SX webcam prediction'
frame_buffer = deque(maxlen=webcam_config.window_size)
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

try:
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands_detector:
        if webcam_config.use_body_pose:
            pose_context = mp_pose.Pose(
                static_image_mode=True,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5,
            )
        else:
            pose_context = None

        try:
            while True:
                success, frame = camera.read()
                if not success:
                    print('Could not read a webcam frame.')
                    break

                frame = cv2.flip(frame, 1)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                feature_vector, detected_hands = extract_rgb(
                    rgb,
                    two_hands=webcam_config.two_hands,
                    use_body_pose=webcam_config.use_body_pose,
                    hands=hands_detector,
                    pose=pose_context,
                )

                if detected_hands:
                    frame_buffer.append(feature_vector)
                    buffered_frames = list(frame_buffer)
                    if len(buffered_frames) < webcam_config.window_size:
                        buffered_frames = (
                            [buffered_frames[-1]] * (webcam_config.window_size - len(buffered_frames))
                            + buffered_frames
                        )
                    input_array = np.stack(buffered_frames, axis=0).astype(np.float32)
                    input_tensor = torch.from_numpy(input_array).unsqueeze(0).to(device)

                    with torch.inference_mode():
                        probabilities = webcam_model(input_tensor).softmax(dim=1)
                        confidence, prediction_index = probabilities.max(dim=1)
                    predicted_word = idx_to_class_webcam[int(prediction_index.item())]
                    display_text = f'{predicted_word} ({confidence.item():.1%})'

                    for hand_landmarks in detected_hands:
                        mp_drawing.draw_landmarks(
                            frame,
                            hand_landmarks,
                            mp_hands.HAND_CONNECTIONS,
                        )
                else:
                    frame_buffer.clear()
                    display_text = 'No hand detected'

                cv2.putText(
                    frame,
                    display_text,
                    (20, 45),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA,
                )
                cv2.imshow(window_name, frame)
                key = cv2.waitKey(1) & 0xFF
                if key in (ord('q'), ord('Q'), 27):
                    break
                try:
                    if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                        break
                except cv2.error:
                    pass
        finally:
            if pose_context is not None:
                pose_context.close()
finally:
    camera.release()
    cv2.destroyAllWindows()

print('Webcam prediction stopped.')